# Async in python

## Coroutine objects

In [ ]:
import asyncio

In [ ]:
async def fetch_data():
    print ("started")

In [ ]:
fetch_data()

In [ ]:
coro = fetch_data()

In [ ]:
await coro

In [ ]:
async def fetch_data():
    print("A")
    print("B")

In [ ]:
coro = fetch_data() # wont be executed because creating the coroutine object does not start it.

print("C")

In [ ]:
async def fetch_data():
    print("A")
    print("B")

coro = fetch_data() # wont be executed yet

print("C") # 1st

await coro # 2nd

print("D") # 3rd

In [ ]:
async def some_io(name, delay):
    print(f"{name}: starting I/O")
    await asyncio.sleep(delay)
    print(f"{name}: I/O completed")

async def A ():
    print ("A1")
    await some_io("A", 3)
    print("A2")

async def B ():
    print ("B1")
    await some_io("B", 1)
    print("B2")
# Note we can have both A and B in one thread

In [ ]:
coro_A  = A() # coroutine objects means "run A and wait until its finished"
coro_B  = B() # after A finished "now run B"

In [ ]:
result_A  = await coro_A # The value returned by the coroutine
result_B  = await coro_B 

``` text
await A()
   │
   ├── A1
   ├── A starts I/O
   ├── A waits 3 seconds
   ├── A I/O completes
   └── A2
          │
          ↓
     await B()
          │
          ├── B1
          ├── B starts I/O
          ├── B waits 1 second
          ├── B I/O completes
          └── B2
```

## Scheduling 

In [ ]:
# task_A, task_b are awaitable 
task_A = asyncio.create_task(A()) # Means schedule A, it does not create a thread. It schedules the coroutine as a Task on the event loop.
task_B = asyncio.create_task(B()) # Schedule B as another Task on the event loop.

In [ ]:
type(task_A)

``` text
Event Loop
│
├── Task A → running/waiting
└── Task B → running/waiting
```
``` text
A1
A: starting I/O
      ↓
A waits 3s
      ↓
B1
B: starting I/O
      ↓
B waits 1s
      ↓
B's I/O completes
      ↓
B2
      ↓
A's I/O completes
      ↓
A2
```

## Event loop

In [ ]:
async def main():
    print("Hello")

In [ ]:
main() # create a coroutine object, nothing drives it.

In [ ]:
# asyncio.run(main()) # wont run in Jupiter as it has already running loop, but it is essential for starting event loop in .py

In [ ]:
async def A():
    print("A1")
    await asyncio.sleep(2)
    print("A2")
async def main():
    task = asyncio.create_task(A())

In [ ]:
await main() # In .py file: asyncio.run(main())

In [ ]:
async def A():
    print ("A1")
    await asyncio.sleep(3)
    print ("A2")

async def main():
    task = asyncio.create_task(A())

    print ("B")

    await task # here `main` says: I need task A to finish before I can continue 

    print ("B")

In [ ]:
await main()

## asyncio.gather()

In [ ]:
async def A():
    print ("A")

async def B():
    print ("B")

async def C():
    print ("C")

In [ ]:
task_A = asyncio.create_task(A())
task_B = asyncio.create_task(B())


In [ ]:
await task_A
await task_B

In [ ]:
results = await asyncio.gather( # means "Run these awaitables concurrently and give me their results when they have all completed"
    A(), 
    B(), 
    C() 
)

In [ ]:
async def A():
    await asyncio.sleep(3)
    print("A")
    return "A"

async def B():
    await asyncio.sleep(1)
    print("B")
    return "B"

async def C():
    await asyncio.sleep(2)
    print("C")
    return "C"

In [ ]:
results = await asyncio.gather(  
    A(), 
    B(), 
    C() 
)


In [ ]:
print(results)

## cancel()

In [ ]:
async def work():
    print ("1: started") # 1

    try:
        await asyncio.sleep(10)
        print("2: finished sleeping")

    except asyncio.CancelledError:
        print("3: cancellation error") # 3

async def main():
    task = asyncio.create_task(work())

    await asyncio.sleep(1)

    print("4: cancelling") # 2
    task.cancel()

    await task 

    print("5: main finished") # 4

```text
create_task(work())
        ↓
Task is scheduled
        ↓
event loop runs work()
        ↓
"1: started"
        ↓
await asyncio.sleep(10)
        ↓
work() suspends
        ↓
main() continues
        ↓
after 1 second:
"4: cancelling task"
        ↓
task.cancel()
        ↓
cancellation is delivered into work()
        ↓
CancelledError occurs at the suspended await
        ↓
except asyncio.CancelledError runs
        ↓
"3: cancellation received"
        ↓
work() finishes
        ↓
await task completes
        ↓
"5: main finished"
```

In [ ]:
await main()

In [ ]:
# Without try except
import asyncio 

async def work():
    print ("1: started") # 1


    await asyncio.sleep(10)
    print("2: finished sleeping")



async def main():
    task = asyncio.create_task(work())

    await asyncio.sleep(1)

    print("4: cancelling") # 2
    task.cancel() 

    await task 

    print("5: main finished")

In [ ]:
await main() # without the try except the cancellation raises CancelledError

In [ ]:
async def work():
    try:
        print ("started")
        await asyncio.sleep(5)
        print ("finished")

    except asyncio.CancelledError:
        print ("cancelled")
        raise
        print("after raise") # Wont get printed as it is after raise

In [ ]:
await work()

In [ ]:
task = asyncio.create_task(work())

In [ ]:
task.cancel()

In [ ]:
async def work():
    try:
        print ("started")
        await asyncio.sleep(10)
        print ("finished")

    except asyncio.CancelledError:
        print ("cancelled")

async def main():
    task = asyncio.create_task(work())

    await asyncio.sleep(1)

    task.cancel()
    print("after cancel")

    await task

    print("main finished")

In [ ]:
await main()

#### Canceling parent 


In [ ]:
async def child():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("chid cancelled")
        raise

async def parent():
    task = asyncio.create_task(child())

    try:
        await task 
    except asyncio.CancelledError:
        print("parent cancelled")
        raise


In [ ]:
parent_task = asyncio.create_task(parent())

In [ ]:
parent_task.cancel()

In [ ]:
await parent_task

```text
parent receives cancellation request
        ↓
parent is currently awaiting child
        ↓
child is cancelled
        ↓
child receives CancelledError
        ↓
child finishes
        ↓
parent receives CancelledError
        ↓
parent finishes
```

In [ ]:
async def child():
    try:
        await asyncio.sleep(4)
        print("child finished")
    except asyncio.CancelledError:
        print("child cancelled")
        raise

async def parent():
    task = asyncio.create_task(child())
    try:
        await asyncio.sleep(1)
    except asyncio.CancelledError:
        print("parent cancelled")
        raise # means raise the same exception again after I have handled it.

In [ ]:
parent_task = asyncio.create_task(parent())


In [ ]:
parent_task.cancel()

In [ ]:
async def child():
    try:
        await asyncio.sleep(5)
        print("child finished")
    except asyncio.CancelledError:
        print("child cancelled")
        raise

async def parent():
    task = asyncio.create_task(child())

    await asyncio.sleep(1)

    task.cancel()

    print("parent continues")

    await task

In [ ]:
parent_task = asyncio.create_task(parent())

#### gather() and cancel()

In [ ]:
async def A():
    try: 
        await asyncio.sleep(2)
        print("A finished")
        return ("A finished")
    except asyncio.CancelledError:
        print("A cancelled")
        raise

async def B():
    try: 
        await asyncio.sleep(4)
        print("B finished")
        return ("B finished")
    except asyncio.CancelledError:
        print("B cancelled")
        raise

In [ ]:
await asyncio.gather(A(),B())

In [ ]:
async def parent():
    await asyncio.gather(A(),B())

parent_task = asyncio.create_task(parent())

await asyncio.sleep(1)

parent_task.cancel()

In [ ]:
async def A():
    
    raise ValueError

async def B():
    try: 
        await asyncio.sleep(4)
        print("B finished")
        return ("B finished")
    except asyncio.CancelledError:
        print("B cancelled")
        raise

In [ ]:
results = await asyncio.gather(
    A(),
    B(),
    return_exceptions=True
) 

In [ ]:
print(results)

## Times out

#### `asyncio.wait_for()`

In [ ]:
# wait_for() and cancellation
async def slow_operation():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("operation cancelled")
        raise
    

In [ ]:
await asyncio.wait_for(slow_operation(), timeout=2) # Time out error not cancelled error

In [ ]:
# Suppressing CancelledError
async def slow_operation():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("operation cancelled")
        # no raise
    

In [ ]:
await asyncio.wait_for(slow_operation(), timeout=2) #  cancelled error bec there is no raise

#### `asyncio.timeout()`

In [ ]:
# Instead of await asyncio.wait_for (work(),timeout = 2)

async with asyncio.timeout(2):
    await slow_operation()

In [ ]:
async def work():
    try: 
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("cleanup")
        raise

In [ ]:
try:
    await asyncio.wait_for(work(), timeout=2)
except TimeoutError:
    print("timed out")

```text
work()
  ↓
await asyncio.sleep(10)
  ↓
2 seconds pass
  ↓
wait_for() requests cancellation
  ↓
CancelledError delivered to work()
  ↓
except asyncio.CancelledError
  ↓
print("cleanup")
  ↓
raise
  ↓
cancellation propagates back to wait_for()
  ↓
wait_for() converts the timeout situation into TimeoutError
  ↓
except TimeoutError
  ↓
print("timed out")
```

In [ ]:
async def get_user():
    await asyncio.sleep(1)
    return "User name"

async def get_preferences():
    await  asyncio.sleep(2)
    return "Good performance"

async def get_recommendations():
    await  asyncio.sleep(4)
    return "Recommendation is 1,2,3"


In [ ]:
print(await get_user())
print(await get_preferences())
print(await get_recommendations())


In [ ]:
# Sequential operations with a timeout
async with asyncio.timeout(5): # Everything inside the block has 5 second deadline.
    await get_user()
    await get_preferences()
    await get_recommendations()

In [ ]:
# Concurrent operations with gather() and timeout.
async with asyncio.timeout(5): 
    result = await asyncio.gather (
        get_user(),
        get_preferences(),
        get_recommendations(),
    )

print (result) # All succeeded.

In [ ]:
# Timeout while gather() is still running,
async with asyncio.timeout(4): 
    result = await asyncio.gather (
        get_user(),
        get_preferences(),
        get_recommendations(),
    )

print (result) # get_recommendations failed (5 sec and max timeout is 4).
                # 1 child operation fails the whole gather fail.

## Async generators and streaming

In [ ]:
# async generator
async def async_numbers():
    yield 1 # yield produces a value and suspends the generator, preserving its state so it can resume later.
    yield 2
    yield 3    

In [ ]:
async for number in async_numbers():
    print(number)

workflow:

```text
async generator
      ↓
request next value
      ↓
yield 1
      ↓
print(1)
      ↓
request next value
      ↓
yield 2
      ↓
print(2)
      ↓
request next value
      ↓
yield 3
      ↓
print(3)
```

In [ ]:
result = async_numbers() # result is async generator object, not the values themselves
print(result)

In [ ]:
# normal generator
def normal_numbers():
    yield 1 
    yield 2
    yield 3    

In [ ]:
for number in normal_numbers():
    print(number)

In [ ]:
async def await_numbers():
    await asyncio.sleep(1)
    yield 1 

    await asyncio.sleep(1)
    yield 2

    await asyncio.sleep(1)
    yield 3    

In [ ]:
async for number in await_numbers():
    print(number)

In [ ]:
# async iterator

async def async_iterator_numbers():
    for i in range(3):
        await asyncio.sleep(1)
        yield i

In [ ]:
async for number in async_iterator_numbers():
    print (number)

In [ ]:
# Coroutine operator

async def coroutine_numbers():
    await asyncio.sleep(3)
    return [0,1,2]

In [ ]:
result = await coroutine_numbers()
result

In [ ]:
# async iterator with sleep
async for number in async_iterator_numbers():
    print(number)
    await asyncio.sleep(5)



async iterator with sleep workflow

```text
t=0   consumer asks for first value
      ↓
      generator sleeps 1s
t=1   generator yields 0
      ↓
      consumer prints 0
      consumer sleeps 5s
      ↓
t=6   consumer asks generator for next value
      ↓
      generator sleeps 1s
t=7   generator yields 1
      ↓
      consumer prints 1
      consumer sleeps 5s
```

## Backpressure

In [ ]:
async def producer(queue):
    for i in range(3):
        await asyncio.sleep(1)
        print(f"Producer: {i}")
        await queue.put(i) # Means put this item into the queue, if there is a room, if not wait.

async def consumer(queue):
    while True:
        item = await queue.get() # The consumer is waiting here.
        print(f"Consumed: {item}")

async def main():
    queue = asyncio.Queue()

    producer_task = asyncio.create_task(producer(queue))
    consumer_task = asyncio.create_task(consumer(queue))

    await producer_task

    # Stop the consumer after the producer finishes
    consumer_task.cancel()

await main()

concept:

```text
queue.put()
    ↓
"Is there space?"
    ↓
YES → put item and continue
NO  → suspend producer until space exists


queue.get()
    ↓
"Is there an item?"
    ↓
YES → get item and continue
NO  → suspend consumer until an item exists
```

In [ ]:
async def producer(queue):
    for i in range(5):
        await asyncio.sleep(1)
        print(f"Producer: {i}")
        await queue.put(i) # Means put this item into the queue, if there is a room, if not wait.
        print(f"Put {i} into queue")

async def consumer(queue):
    for _ in range(5):
        item = await queue.get() # The consumer is waiting here.
        print(f"Consumed: {item}")
        await asyncio.sleep(3)

async def main():
    queue = asyncio.Queue(maxsize=2) # Tiny capacity

    producer_task = asyncio.create_task(producer(queue))
    consumer_task = asyncio.create_task(consumer(queue))

    await producer_task

    # Stop the consumer after the producer finishes
    consumer_task.cancel()

await main()

## Async context manger

In [ ]:
# Manually acquire resources

file = open("data.txt")

# use the file
data = file.read()

file.close()

data

In [ ]:
# Resource leak

f = open("data.txt")

# use the file
data = f.read()

# forgot:
# f.close()

In [ ]:
# Acquire resources using context manger.


with open("data.txt") as f:
    data = f.read()

data

Workflow of `with`:

```text
open("data.txt")
      ↓
file object
      ↓
__enter__()
      ↓
returns the object used as `f`
      ↓
run:
    data = f.read()
      ↓
__exit__()
      ↓
file gets closed
```

In [ ]:
data = "empty" # over wright the value of data  
file = open("data.txt")

data = file.read()

process(data)   # this raises an exception

file.close()

In [ ]:
data # data still got processed BUT the resource ("data.txt") is still open

In [ ]:
data = "empty" # over wright the value of data  

with open("data.txt") as f:
    data = f.read()
    process(data) # exception

In [ ]:
data # data still got processed AND the resource ("data.txt") got closed

In [ ]:
# Tiny custom context manger

class MyResource:
    def __enter__(self):
        print("Entering")
        return self
    
    def __exit__(self, exc_type, exc, tb):
        print ("Exiting")

In [ ]:
with MyResource() as resource:
    print("Using resource")

Workflow:

```text
MyResource created
       ↓
__enter__()
       ↓
"Entering"
       ↓
resource = returned value
       ↓
"Using resource"
       ↓
__exit__()
       ↓
"Exiting"
```

`with` is not doing the resource management itself.

In [ ]:
class Resource:
    def __enter__(self):
        print("ENTER")
        return "hello"
    
    def __exit__(self, exc_type, exc, tb):
        print ("EXIT")

In [ ]:
with Resource() as x:
    print(x)

In [ ]:
class Resource:
    def __enter__(self):
        print("ENTER")
        return self
    
    def __exit__(self, exc_type, exc, tb):
        print ("EXIT")

In [ ]:
with Resource() as x:
    print("INSIDE")
    raise ValueError ("Something went wrong")

Print ("After")

In [ ]:
# Without the __exit__ arguments

class Resource:
    def __enter__(self):
        print("ENTER")
        return self
    
    def __exit__(self):
        print ("EXIT")


In [ ]:
with Resource() as x:
    print("INSIDE")
    raise ValueError ("Something went wrong")

Print ("After")
# Notice that `EXIT` hasn't been printed because python cannot successfully invoke __exit__ args.
# The method only accept `self` but the context manager protocol supplies `self`, `exc_type`, `exc_value`, and `traceback`.

In [ ]:
class Resource:
    def __enter__(self):
        print("ENTER")
        return "Finished"
    
    def __exit__(self, exc_type, exc_value, traceback):
        print ("EXIT")
        print("type:", exc_type )
        print("value:", exc_value )
        print("traceback:", traceback )

In [ ]:
with Resource():
    print("INSIDE")
    raise ValueError("Something went wrong")

#### Lifecycle of a `with` statement

In [ ]:
with Resource() as x:
    print(x)

In [ ]:
# Can be written as (but not the literal implementation, its is just useful mental model)

"""
resource = Resource()

x = resource.__enter__()

try:
    print(x)
finally:
    resource.__exit__(...)
    """

NameError: name 'Resource' is not defined

In [ ]:
class Connection:
    def __enter__(self):
        print("Connection opened")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Connection closed")

with Connection() as conn:
    print("Doing work")

In [ ]:
class Resource:
    def __enter__(self):
        print("ENTER")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("EXIT")
        print("Exception type:", exc_type)
        print("Exception value:", exc_value)

In [ ]:
with Resource():
    print("INSIDE")
    raise ValueError("Something went wrong")

In [ ]:
# __exit__ with return True

class Resource:
    def __enter__(self):
        print("ENTER")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("EXIT")
        return True
    

In [ ]:
with Resource():
    print("INSIDE")
    raise ValueError("Something went wrong")

print("AFTER")

In [ ]:
class Resource:
    def __enter__(self):
        print("ENTER")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("EXIT")
        print("Exception type:", exc_type)
        print("Exception value:", exc_value)
        return True
    

In [ ]:
with Resource():
    print("INSIDE")
    raise ValueError("Something went wrong")

print("AFTER")
# The error happened but returning `True` prevented python from propagate it.

### Simple context manager

In [3]:
class Connection:
    def __enter__(self):
        print("Opening connection")
        return self
    
    def send(self,message):
        print (f"Sending: {message}")

    def __exit__(self, exc_type, exc_value, traceback):
        print ("Closing connection")
        

In [4]:
with Connection() as conn:
    conn.send("Hello")
    conn.send("there")

Opening connection
Sending: Hello
Sending: there
Closing connection


In [8]:
# Error handling

class Connection:
    def __enter__(self):
        print("Opening connection")
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        print ("Closing connection")

    def send(self):
        print (f"Sending")
        raise ValueError("Connection failed")

In [ ]:
with Connection() as conn:
    conn.send()

print("After") # "After" wont be printed as the valueError propagates (reaches) outside the block
# But the resource gets cleaned up (closed) even thought the operation using it failed.

Opening connection
Sending
Closing connection


ValueError: Connection failed

### Actual `open()` example

In [11]:
with open("data.txt") as f:
    data = f.read()

The Lifecycle:

```text
open("data.txt")
       ↓
file object
       ↓
__enter__()
       ↓
returned value → f
       ↓
f.read()
       ↓
leave with block
       ↓
__exit__()
       ↓
file is closed
```

In [ ]:
# Error handling

with open("data.txt") as f:
    data = f.read()
    process(data) # raises an exception

NameError: name 'process' is not defined

The lifecycle with the error still:

```text
open file
   ↓
__enter__()
   ↓
f = returned file object
   ↓
read/process
   ↓
exception
   ↓
__exit__()
   ↓
file closed
   ↓
exception propagates
```
